# MMVC Trainer - Training Notebook

This notebook provides a simple interface to train MMVC models using the CLI scripts.
Perfect for Google Colab or Jupyter notebook environments.

## Setup

First, let's install the required dependencies and setup the environment.

In [ ]:
# Install dependencies
!pip install torch torchaudio
!pip install librosa scipy numpy
!pip install pyopenjtalk
!pip install tensorboard tqdm matplotlib
!pip install onnx onnxruntime

# Install additional dependencies
!pip install resampy unidecode psutil
!pip install protobuf==3.20.1

In [ ]:
# Clone the repository (if running on Colab)
import os
if not os.path.exists('MMVC_Trainer-my'):
    !git clone https://github.com/your-username/MMVC_Trainer-my.git
    
os.chdir('MMVC_Trainer-my')
print("Current directory:", os.getcwd())

## Configuration

Set up your training configuration.

In [ ]:
# Configuration paths
CONFIG_PATH = "configs/base_config.json"
MODEL_DIR = "models/experiment_1"
DATA_DIR = "data/processed"

# Check if config file exists
if os.path.exists(CONFIG_PATH):
    print(f"Configuration file found: {CONFIG_PATH}")
else:
    print(f"Configuration file not found: {CONFIG_PATH}")
    print("Please make sure the config file exists.")

# Create model directory
os.makedirs(MODEL_DIR, exist_ok=True)
print(f"Model directory: {MODEL_DIR}")

## Data Preprocessing

First, let's preprocess the data and create the training file lists.

In [ ]:
# Data preprocessing
RAW_DATA_DIR = "data/raw"
PROCESSED_DATA_DIR = "data/processed"

# Run preprocessing
!python -m src.cli.preprocess \
    --input-dir {RAW_DATA_DIR} \
    --output-dir {PROCESSED_DATA_DIR} \
    --config {CONFIG_PATH} \
    --verbose

## Training

Now let's start training the model.

In [ ]:
# Start training
!python -m src.cli.train \
    --config {CONFIG_PATH} \
    --model {MODEL_DIR}

## Monitor Training

You can monitor training progress using TensorBoard.

In [ ]:
# Load TensorBoard in Jupyter
%load_ext tensorboard
%tensorboard --logdir {MODEL_DIR}/logs

## Voice Conversion

After training, you can perform voice conversion.

In [ ]:
# Voice conversion example
INPUT_AUDIO = "path/to/input/audio.wav"
OUTPUT_AUDIO = "path/to/output/converted.wav"
CHECKPOINT = f"{MODEL_DIR}/G_100000.pth"  # Use your trained checkpoint
TARGET_SPEAKER = 0  # Target speaker ID

!python -m src.cli.convert \
    --config {CONFIG_PATH} \
    --checkpoint {CHECKPOINT} \
    --input {INPUT_AUDIO} \
    --output {OUTPUT_AUDIO} \
    --target-speaker {TARGET_SPEAKER} \
    --verbose

## Model Export

Export your trained model to ONNX format for deployment.

In [ ]:
# Export to ONNX
ONNX_OUTPUT = f"{MODEL_DIR}/model.onnx"

!python -m src.cli.export \
    --config {CONFIG_PATH} \
    --checkpoint {CHECKPOINT} \
    --output {ONNX_OUTPUT} \
    --dynamic-axes \
    --optimize \
    --verbose

## Utilities

Some utility functions for working with the models.

In [ ]:
import json
import torch
import librosa
import matplotlib.pyplot as plt
import IPython.display as ipd

def load_config(config_path):
    """Load configuration file."""
    with open(config_path, 'r', encoding='utf-8') as f:
        return json.load(f)

def plot_spectrogram(audio_path, sr=22050):
    """Plot spectrogram of audio file."""
    y, sr = librosa.load(audio_path, sr=sr)
    S = librosa.stft(y)
    S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)
    
    plt.figure(figsize=(12, 6))
    librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='hz')
    plt.colorbar(format='%+2.0f dB')
    plt.title('Spectrogram')
    plt.tight_layout()
    plt.show()

def play_audio(audio_path, sr=22050):
    """Play audio file in notebook."""
    y, sr = librosa.load(audio_path, sr=sr)
    return ipd.Audio(y, rate=sr)

# Load and display config
if os.path.exists(CONFIG_PATH):
    config = load_config(CONFIG_PATH)
    print("Current configuration:")
    print(json.dumps(config, indent=2))

## Tips for Google Colab

If you're running this on Google Colab:

1. Make sure to enable GPU runtime (Runtime → Change runtime type → GPU)
2. Mount Google Drive to save your models:
   ```python
   from google.colab import drive
   drive.mount('/content/drive')
   ```
3. Copy your trained models to Google Drive to avoid losing them:
   ```bash
   !cp -r models/experiment_1 /content/drive/MyDrive/MMVC_models/
   ```